In [1]:

# Import pandas for CSV processing.
import pandas as pd

# Import Path for handling file paths.
from pathlib import Path

# Set the path to the raw validation dataset.
raw_file = Path(
    "../../Raw_Data/Disaster_Response_management_Data/"
    "disaster_response_validation.csv"
)

# Set the folder where processed files will be saved.
output_folder = Path(".")

# Create the output folder if it does not exist.
output_folder.mkdir(parents=True, exist_ok=True)

# Load only the validation dataset.
df = pd.read_csv(raw_file, low_memory=False)

# Record the original number of rows and columns.
original_rows = len(df)
original_columns = len(df.columns)

# Count missing values before cleaning.
missing_before = int(df.isna().sum().sum())

# Remove columns that are completely empty.
empty_columns = df.columns[df.isna().all()].tolist()
df = df.drop(columns=empty_columns)

# Clean surrounding spaces in text columns without changing missing values.
text_columns = ["message", "original", "genre", "split", "event"]

for column in text_columns:
    if column in df.columns:
        df[column] = df[column].map(
            lambda value: value.strip()
            if isinstance(value, str)
            else value
        )

# Convert the date column to datetime format if present.
if "date_haiti" in df.columns:
    df["date_haiti"] = pd.to_datetime(
        df["date_haiti"],
        errors="coerce"
    )

# Count exact duplicate rows before removing them.
duplicate_rows = int(df.duplicated().sum())

# Remove exact duplicate rows, keeping the first occurrence.
df = df.drop_duplicates().reset_index(drop=True)

# Count repeated IDs remaining after exact duplicate removal.
duplicate_ids = (
    int(df["id"].duplicated().sum())
    if "id" in df.columns else 0
)

# Count missing message text.
missing_messages = (
    int(df["message"].isna().sum())
    if "message" in df.columns else 0
)

# Count missing values after cleaning.
missing_after = int(df.isna().sum().sum())

# Save the processed validation dataset.
output_file = output_folder / "disaster_response_validation_processed.csv"
df.to_csv(output_file, index=False)

# Create a report containing this file's preprocessing results.
report = pd.DataFrame([{
    "Original_Rows": original_rows,
    "Processed_Rows": len(df),
    "Original_Columns": original_columns,
    "Processed_Columns": len(df.columns),
    "Empty_Columns_Removed": (
        ", ".join(empty_columns) if empty_columns else "None"
    ),
    "Duplicate_Rows_Removed": duplicate_rows,
    "Duplicate_ID_Count": duplicate_ids,
    "Missing_Messages": missing_messages,
    "Missing_Values_Before": missing_before,
    "Missing_Values_After": missing_after
}])

# Save the validation preprocessing report separately.
report_file = output_folder / "disaster_response_validation_preprocessing_report.csv"
report.to_csv(report_file, index=False)

# Display the validation preprocessing results.
print("--- VALIDATION DATA PREPROCESSING RESULTS ---")
print("Original shape:", (original_rows, original_columns))
print("Processed shape:", df.shape)
print("Empty columns removed:", empty_columns)
print("Duplicate rows removed:", duplicate_rows)
print("Duplicate IDs remaining:", duplicate_ids)
print("Missing messages:", missing_messages)
print("Missing values before:", missing_before)
print("Missing values after:", missing_after)
print("\nProcessed CSV saved to:", output_file.resolve())
print("Preprocessing report saved to:", report_file.resolve())

# Display the saved report as a table.
display(report)


--- VALIDATION DATA PREPROCESSING RESULTS ---
Original shape: (2337, 45)
Processed shape: (2335, 45)
Empty columns removed: []
Duplicate rows removed: 2
Duplicate IDs remaining: 2
Missing messages: 0
Missing values before: 5384
Missing values after: 5376

Processed CSV saved to: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\Disaster Response Messages\disaster_response_validation_processed.csv
Preprocessing report saved to: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\Disaster Response Messages\disaster_response_validation_preprocessing_report.csv


,Original_Rows,Processed_Rows,Original_Columns,Processed_Columns,Empty_Columns_Removed,Duplicate_Rows_Removed,Duplicate_ID_Count,Missing_Messages,Missing_Values_Before,Missing_Values_After
0,2337,2335,45,45,None,2,2,0,5384,5376
